## 과제 B: 날짜 계산기 (실무 빈도 ★★★★☆)

```python
class DateDiff(BaseModel):
    date1: str
    date2: str
    days: int
    weeks: int

@tool
def days_between(date1: str, date2: str) -> DateDiff:
    """두 날짜(YYYY-MM-DD) 사이의 일수·주수를 반환합니다."""
    ...
```

In [1]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")

# 3️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")

from typing import Literal, Optional
from pydantic import BaseModel, Field   # 1-2교시 정형화 도구
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool   # ⭐ 오늘의 주인공

# Tool Calling은 temperature=0 권장 (도구 선택의 일관성)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅ import 완료")

✅ 로컬 Jupyter
✅ API 키 OK
✅ import 완료


In [3]:
def run_with_tools(question: str, tools: list, llm_obj=llm, max_steps: int = 5):
    """도구 호출이 끝날 때까지 반복 실행하고 최종 답변 반환"""
    llm_with_tools = llm_obj.bind_tools(tools)
    tool_map = {t.name: t for t in tools}
    messages = [HumanMessage(content=question)]

    for _ in range(max_steps):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:          
            return response.content

        for tc in response.tool_calls:
            result = tool_map[tc["name"]].invoke(tc["args"])

            error = getattr(result, "error", None)
            if error:
                return str(error)

            messages.append(
                ToolMessage(content=str(result), tool_call_id=tc["id"])
            )

    return "최대 단계 수를 초과했습니다."

print("✅ 헬퍼 함수 준비 완료")

✅ 헬퍼 함수 준비 완료


In [5]:
from datetime import datetime

In [6]:
class DateDiff(BaseModel):
    date1: str
    date2: str
    days: int
    weeks: int
    error: Optional[str] = None

@tool
def days_between(date1: str, date2: str) -> DateDiff:
    """두 날짜(YYYY-MM-DD) 사이의 일수·주수를 반환합니다."""

    start = datetime.strptime(date1, "%Y-%m-%d")
    end = datetime.strptime(date2, "%Y-%m-%d")
    days = (end - start).days
    weeks = days // 7

    return DateDiff(date1=date1, date2=date2, days=days, weeks=weeks)

In [ ]:
q = ["2023-01-01과 2023-12-31 사이의 일수와 주수는?", "2024-02-29와 2024-03-01 사이의 일수와 주수는?",]
for question in q:
    answer = run_with_tools(question, [days_between])
    print(f"질문: {question}\n답변: {answer}\n")

질문: 2023-01-01과 2023-12-31 사이의 일수와 주수는?
답변: 2023-01-01과 2023-12-31 사이의 일수는 364일이며, 주수는 52주입니다.

질문: 2024-02-29와 2024-03-01 사이의 일수와 주수는?
답변: 2024-02-29와 2024-03-01 사이의 일수는 1일이며, 주수는 0주입니다.



: 